# VoiceMood — One-time Embedding Extraction (Colab)

This notebook does the only step in the project that needs a GPU.
Run it **once**. It produces a small file (`features.npz`, about 10 MB)
that the laptop-side code uses for everything else.

## What it does
1. Downloads RAVDESS speech audio (~250 MB).
2. Loads two pretrained speech models:
   - **wav2vec2-base** (PyTorch / HuggingFace)
   - **YAMNet** (TensorFlow / TF Hub)
3. Extracts a single embedding per clip from each model.
4. Saves embeddings + labels to `features.npz` and downloads it.

## Setup
Make sure you've selected a GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!pip install -q transformers torchaudio librosa tensorflow tensorflow_hub tqdm

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

WORK = Path('/content/voicemood_work')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

## 1. Download RAVDESS

In [ ]:
import urllib.request
import zipfile

URL = 'https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip'
ZIP_PATH = WORK / 'ravdess.zip'
RAW_DIR = WORK / 'raw'
RAW_DIR.mkdir(exist_ok=True)

if not ZIP_PATH.exists():
    print('Downloading RAVDESS (~250 MB)...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print('Done.')

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(RAW_DIR)

wavs = sorted(RAW_DIR.rglob('*.wav'))
print(f'Found {len(wavs)} wav files.')

## 2. Parse RAVDESS filenames

RAVDESS filename schema: `mod-channel-EMOTION-int-stmt-rep-actor.wav`.
We only use audio-speech (`03-01-...`). Emotion is the 3rd token.

In [ ]:
EMOTION_CODES = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised',
}
LABELS = list(EMOTION_CODES.values())
LABEL_TO_IDX = {lab: i for i, lab in enumerate(LABELS)}

valid = []
for w in wavs:
    parts = w.stem.split('-')
    if len(parts) != 7:
        continue
    modality, channel, emo, _, _, _, actor = parts
    if modality != '03' or channel != '01' or emo not in EMOTION_CODES:
        continue
    valid.append((w, LABEL_TO_IDX[EMOTION_CODES[emo]], int(actor)))

print(f'Valid clips: {len(valid)}')
paths = [v[0] for v in valid]
labels = np.array([v[1] for v in valid], dtype=np.int64)
actors = np.array([v[2] for v in valid], dtype=np.int64)

## 3. wav2vec2 embeddings (PyTorch)

We use the base model with frozen weights and mean-pool the last hidden state across time.
Output dim: 768 per clip.

In [ ]:
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

MODEL_ID = 'facebook/wav2vec2-base'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
w2v = Wav2Vec2Model.from_pretrained(MODEL_ID).to(device).eval()
for p in w2v.parameters():
    p.requires_grad = False
print('Loaded wav2vec2-base on', device)

In [ ]:
SR = 16000
MAX_LEN = SR * 5  # cap at 5 seconds to keep memory predictable

wav2vec_embs = []
for path in tqdm(paths, desc='wav2vec2'):
    audio, _ = librosa.load(str(path), sr=SR, mono=True)
    audio = audio[:MAX_LEN]
    inputs = extractor(audio, sampling_rate=SR, return_tensors='pt', padding=True)
    input_values = inputs['input_values'].to(device)
    with torch.no_grad():
        out = w2v(input_values)
    # Mean-pool the last hidden state across the time axis.
    emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy().astype(np.float32)
    wav2vec_embs.append(emb)

wav2vec_embs = np.stack(wav2vec_embs)
print('wav2vec2 embeddings:', wav2vec_embs.shape)

## 4. YAMNet embeddings (TensorFlow)

YAMNet is Google's pretrained audio event classifier; we use its 1024-dim embedding layer.
Mean-pooled across time.

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
print('Loaded YAMNet')

In [ ]:
yamnet_embs = []
for path in tqdm(paths, desc='YAMNet'):
    audio, _ = librosa.load(str(path), sr=16000, mono=True)
    audio = audio[:MAX_LEN]
    _, emb_seq, _ = yamnet(audio.astype(np.float32))
    emb = tf.reduce_mean(emb_seq, axis=0).numpy().astype(np.float32)
    yamnet_embs.append(emb)

yamnet_embs = np.stack(yamnet_embs)
print('YAMNet embeddings:', yamnet_embs.shape)

## 5. Save and download

The next cell downloads `features.npz` to your computer. Place it under
`voicemood/data/embeddings/` in your local project.

In [ ]:
OUT = WORK / 'features.npz'
np.savez_compressed(
    OUT,
    wav2vec2=wav2vec_embs,
    yamnet=yamnet_embs,
    labels=labels,
    actors=actors,
)
print(f'Saved {OUT} — {OUT.stat().st_size / 1024 / 1024:.2f} MB')

In [ ]:
from google.colab import files
files.download(str(OUT))

Done. On your laptop:

1. Move the downloaded `features.npz` into `voicemood/data/embeddings/`.
2. Run `python scripts/train_all.py`.
3. Run `streamlit run src/voicemood/app.py` to see the demo.